# Solar & Wind Atlas catalog explorer (offline)

Inspect the bundled layer catalog without touching the network: which layers
ship, which atlas each comes from, and the transport used to fetch it. This is
the no-network companion to the live wind quickstart.


In [1]:
import pandas as pd

from earthlens.solar_wind_atlas import Catalog


## The shipped layers

`Catalog()` loads the bundled sharded `catalog/` (`solar.yaml` + `wind.yaml`).
Each row is a frozen pydantic `Layer` carrying its atlas, transport, URL, and units.


In [2]:
catalog = Catalog()
rows = [
    {
        "id": lid,
        "atlas": catalog.get(lid).atlas,
        "transport": catalog.get(lid).transport,
        "units": catalog.get(lid).units,
        "long_name": catalog.get(lid).long_name,
    }
    for lid in catalog.available()
]
pd.DataFrame(rows)


,id,atlas,transport,units,long_name
0,air_density_100m,gwa,vsicurl,kg/m3,Mean air density at 100 m above ground level
1,capacity_factor_iec1,gwa,vsicurl,fraction,"Wind turbine capacity factor, IEC Class 1"
2,capacity_factor_iec2,gwa,vsicurl,fraction,"Wind turbine capacity factor, IEC Class 2"
3,capacity_factor_iec3,gwa,vsicurl,fraction,"Wind turbine capacity factor, IEC Class 3"
4,dif,gsa,download_zip,kWh/m2/day,Diffuse horizontal irradiation (long-term aver...
5,dni,gsa,download_zip,kWh/m2/day,Direct normal irradiation (long-term average d...
6,ghi,gsa,download_zip,kWh/m2/day,Global horizontal irradiation (long-term avera...
7,gti,gsa,download_zip,kWh/m2/day,Global irradiation for optimally tilted surfac...
8,opta,gsa,download_zip,degree,Optimum tilt of PV modules
9,pvout,gsa,download_zip,kWh/kWp/day,Photovoltaic power potential — specific PV out...


## Two atlases, two transports

- **Global Wind Atlas** layers (`atlas='gwa'`) use the `vsicurl` transport: a
  windowed read straight from the remote Cloud-Optimized GeoTIFF — only the AOI
  transfers.
- **Global Solar Atlas** layers (`atlas='gsa'`) use `download_zip`: the deflate
  ZIP archive (no random access) is downloaded once into a cache, then cropped
  locally.


In [3]:
by_transport = {}
for lid in catalog.available():
    by_transport.setdefault(catalog.get(lid).transport, []).append(lid)
by_transport


{'vsicurl': ['air_density_100m',
  'capacity_factor_iec1',
  'capacity_factor_iec2',
  'capacity_factor_iec3',
  'weibull_k_100m',
  'weibull_k_10m',
  'weibull_k_150m',
  'weibull_k_200m',
  'weibull_k_50m',
  'wind_100m'],
 'download_zip': ['dif', 'dni', 'ghi', 'gti', 'opta', 'pvout']}

## Did-you-mean on an unknown id

An unknown layer id raises a `ValueError` with the closest match.


In [4]:
try:
    catalog.get("gho")
except ValueError as exc:
    print(exc)


'gho' is not in the Solar & Wind Atlas catalog. Known layers: ['air_density_100m', 'capacity_factor_iec1', 'capacity_factor_iec2', 'capacity_factor_iec3', 'dif', 'dni', 'ghi', 'gti', 'opta', 'pvout', 'weibull_k_100m', 'weibull_k_10m', 'weibull_k_150m', 'weibull_k_200m', 'weibull_k_50m', 'wind_100m']. Did you mean 'ghi'?
